In [ ]:
# Inputs: Each fire point's reference data and the GEE feature data
    
# Outputs: DataFrame for ML model training

# Purpose: calculate spectral indices, gap-fill, one-hot encode land cover

In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', None)
import geopandas
import rasterio

In [ ]:
def window_get_point_raster_values(raster, vector):
    vector_new = vector.copy().to_crs(raster.crs) 
    coord_list = [(x, y) for x, y in zip(vector_new["geometry"].x, vector_new["geometry"].y)]
    vector_new["value"] = [x for x in raster.sample(coord_list)]
    return vector_new


In [ ]:
fires_with_factors = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartTwoOutputShape3.shp")
fires_with_factors.head()


In [ ]:
gee_df = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartThreeOutput2.csv")
gee_df.head()

In [ ]:
gee_df.rename(columns={'fire_id': 'FIRE_ID'}, inplace=True)
fires_with_factors.rename(columns={'Occurrence': 'occur_id', 'point_orde': 'point_id'}, inplace=True)

In [ ]:
gee_df = gee_df.astype({"FIRE_ID": "str", "occur_id": "str", "point_id": "str"})
fires_with_factors = fires_with_factors.astype({"FIRE_ID": "str", "occur_id": "str", "point_id": "str"})

In [ ]:
fires_with_factors = pd.merge(fires_with_factors, gee_df, on=['FIRE_ID', 'occur_id', 'point_id'], how='left')

In [ ]:
fires_with_factors['Month'] = pd.to_datetime(fires_with_factors['IG_DATE']).dt.month

In [ ]:
lc = rasterio.open(r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\Land Cover\nlcd_2021_land_cover_test.tif")

In [ ]:
fires_3 = window_get_point_raster_values(lc, fires_with_factors)

In [ ]:
fires_3["LandCover"] = fires_3["value"].apply(lambda x: x[0])

In [ ]:
def get_NDVI(row):
    if row['IG_DATE'] > '2013-05-17':
        NDVI = (row['SR_B5'] - row['SR_B4']) / (row['SR_B5'] + row['SR_B4'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NDVI = (row['SR_B4'] - row['SR_B3']) / (row['SR_B4'] + row['SR_B3'])
    elif row['IG_DATE'] < '1999-07-29':
        NDVI = (row['SR_B4'] - row['SR_B3']) / (row['SR_B4'] + row['SR_B3'])
    return NDVI

def get_NDWI(row):
    if row['IG_DATE'] > '2013-05-17':
        NDWI = (row['SR_B5'] - row['SR_B3']) / (row['SR_B5'] + row['SR_B3'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NDWI = (row['SR_B4'] - row['SR_B2']) / (row['SR_B4'] + row['SR_B2'])
    elif row['IG_DATE'] < '1999-07-29':
        NDWI = (row['SR_B4'] - row['SR_B2']) / (row['SR_B4'] + row['SR_B2'])
    return NDWI

def get_NBR(row):
    if row['IG_DATE'] > '2013-05-17':
        NBR = (row['SR_B5'] - row['SR_B7']) / (row['SR_B5'] + row['SR_B7'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NBR = (row['SR_B4'] - row['SR_B7']) / (row['SR_B4'] + row['SR_B7'])
    elif row['IG_DATE'] < '1999-07-29':
        NBR = (row['SR_B4'] - row['SR_B7']) / (row['SR_B4'] + row['SR_B7'])
    return NBR


In [ ]:
fires_3['th'] = pd.to_numeric(fires_3["th"], errors="coerce")

In [ ]:
fires_3['th'] = fires_3.apply(lambda x: x['th'] if x['th'] <= 180 else x['th']-360, axis=1)
fires_3.head()

In [ ]:
fires_3.info(verbose=True, show_counts=True)

In [ ]:
fires_3 = fires_3.fillna(fires_3.groupby("occur_id").ffill())
fires_3 = fires_3.dropna(thresh=fires_3.shape[1])

In [ ]:
fires_3.info(verbose=True, show_counts=True)

In [ ]:
fires_3["SR_B2"] = pd.to_numeric(fires_3["SR_B2"], errors="coerce")
fires_3["SR_B3"] = pd.to_numeric(fires_3["SR_B3"], errors="coerce")
fires_3["SR_B4"] = pd.to_numeric(fires_3["SR_B4"], errors="coerce")
fires_3["SR_B5"] = pd.to_numeric(fires_3["SR_B5"], errors="coerce")
fires_3["SR_B7"] = pd.to_numeric(fires_3["SR_B7"], errors="coerce")
fires_3['aspect'] = pd.to_numeric(fires_3["aspect"], errors="coerce")
fires_3['hillshade'] = pd.to_numeric(fires_3["hillshade"], errors="coerce")
fires_3['slope'] = pd.to_numeric(fires_3["slope"], errors="coerce")
fires_3['bi'] = pd.to_numeric(fires_3["bi"], errors="coerce")
fires_3['erc'] = pd.to_numeric(fires_3["erc"], errors="coerce")
fires_3['eto'] = pd.to_numeric(fires_3["eto"], errors="coerce")
fires_3['fm100'] = pd.to_numeric(fires_3["fm100"], errors="coerce")
fires_3['fm1000'] = pd.to_numeric(fires_3["fm1000"], errors="coerce")
fires_3['pr'] = pd.to_numeric(fires_3["pr"], errors="coerce")
fires_3['rmax'] = pd.to_numeric(fires_3["rmax"], errors="coerce")
fires_3['rmin'] = pd.to_numeric(fires_3["rmin"], errors="coerce")
fires_3['tmmn'] = pd.to_numeric(fires_3["tmmn"], errors="coerce")
fires_3['tmmx'] = pd.to_numeric(fires_3["tmmx"], errors="coerce")
fires_3['vpd'] = pd.to_numeric(fires_3["vpd"], errors="coerce")
fires_3['vs'] = pd.to_numeric(fires_3["vs"], errors="coerce")
fires_3['pop_density'] = pd.to_numeric(fires_3["pop_density"], errors="coerce")
fires_3['u_class'] = pd.to_numeric(fires_3["u_class"], errors="coerce")


In [ ]:
fires_3['NDVI'] = fires_3.apply(lambda x: get_NDVI(x), axis=1)
fires_3['NDWI'] = fires_3.apply(lambda x: get_NDWI(x), axis=1)
fires_3['NBR'] = fires_3.apply(lambda x: get_NBR(x), axis=1)

In [ ]:
fires_3.head()

fires_3.columns

In [ ]:
print(len(fires_3))
#fires_3 = fires_3[fires_3["IG_DATE"] > '1994-12-31']

In [ ]:
fires_5 = fires_3[['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'WUIBreach', 'Month', 'NDVI', 'NDWI', 'NBR', 'DEM', 'aspect', 'hillshade', 'slope', 'bi', 'erc', 'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'th', 'pop_density', 'tmmn', 'tmmx', 'vpd', 'vs', 'LandCover', "u_class", 'Y', 'geometry']]

In [ ]:
fires_5.head()

In [ ]:
fires_5 = fires_5.fillna(fires_5.groupby("occur_id").ffill())
fires_5["u_class"] = fires_5["u_class"].fillna(-200)

In [ ]:
fires_5 = fires_5.dropna(thresh=fires_5.shape[1])

In [ ]:
fires_5 = pd.get_dummies(fires_5, columns=['u_class'], dtype=int)

In [ ]:
print(len(fires_5))

In [ ]:
first_urban_point = (fires_5[fires_5["IsUrban"] == 1].groupby("occur_id")["point_id"].min())

In [ ]:
fires_5["first_urban_point"] = fires_5["occur_id"].map(first_urban_point)

In [ ]:
fires_5["RemoveFlag"] = (fires_5["point_id"]).astype(int) > ((fires_5["first_urban_point"]).astype(int) + 5)

In [ ]:
fires_5 = fires_5[fires_5["RemoveFlag"] == 0]

In [ ]:
fires_ref = fires_5[['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'WUIBreach', 'geometry', 'Y']]

In [ ]:
fires_5.rename(columns={"u_class_10.0": "Water", "u_class_11.0": "Very_low_rural", "u_class_12.0": "Low_density_rural", "u_class_13.0": "Rural_cluster", "u_class_21.0": "Suburban", "u_class_22.0": "Semi_dense_urban", "u_class_23.0": "Dense_urban", "u_class_30.0": "Urban_centre", "u_class_-200.0": "No_Data_Urban_Class"}, inplace=True)
fires_5.drop(columns=['geometry'], inplace=True)

In [ ]:
fires_5.info(verbose=True, show_counts=True)

In [ ]:
fires_5.to_csv(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartFourOutput_WUI.csv")

In [ ]:
fires_ref.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartFourOutput_ref_WUI.shp")